# 🔗 Impacts indirects — Supply Chain / BoARIO (CLIMADA)

Exploration du moteur BoARIO (Adaptative Regional Input-Output) pour modéliser la propagation des impacts climatiques dans les chaînes d'approvisionnement.

**Approche** : Multi-péril → impacts directs → propagation économique  
**Zone** : Globale (tables MRIO)  
**Données** : EXIOBASE / WIOD / synthétique  
**Métriques** : Coûts indirects, multiplicateurs, effets de cascade

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

try:
    from climada.engine import Impact
    CLIMADA_OK = True
except ImportError:
    CLIMADA_OK = False

try:
    from climada.engine.supplychain import BoARIOModel, DirectShocksSet
    BOARIO_OK = True
except ImportError:
    try:
        from pinar.engine.supplychain import BoARIOModel, DirectShocksSet
        BOARIO_OK = True
    except ImportError:
        BOARIO_OK = False
        print("⚠️ BoARIO non disponible — on utilisera une simulation simplifiée")

print(f"CLIMADA disponible : {CLIMADA_OK}")
print(f"BoARIO disponible : {BOARIO_OK}")

## 1. Concept — Impacts directs vs indirects

Les catastrophes climatiques génèrent deux types de coûts :

| Type | Description | Exemple |
|------|-------------|---------|
| **Direct** | Destruction physique d'actifs | Usine inondée → perte de stock |
| **Indirect** | Perturbation des chaînes économiques | Usine arrêtée → fournisseurs/clients impactés |

### BoARIO (Adaptative Regional Input-Output)
Modèle développé à ETH Zurich qui simule la **propagation dynamique** d'un choc d'offre/demande dans une économie multi-régionale :

1. **Choc initial** : perte de capacité de production (output direct CLIMADA)
2. **Propagation amont** : les fournisseurs perdent des commandes
3. **Propagation aval** : les clients manquent d'inputs
4. **Adaptation** : substitution, stocks, reconstruction
5. **Récupération** : retour progressif à l'équilibre

### Tables MRIO (Multi-Regional Input-Output)
- **EXIOBASE** : 44 pays × 163 secteurs
- **WIOD** : 43 pays × 56 secteurs
- **GTAP** : 141 régions × 65 secteurs

## 2. Simulation simplifiée — Modèle IO statique

En l'absence de BoARIO, on implémente un modèle Input-Output de Leontief simplifié pour illustrer les mécanismes de propagation.

In [ ]:
# --- Modèle Input-Output simplifié (Leontief) ---

# 5 secteurs × 3 régions
sectors = ['Agriculture', 'Industrie', 'Énergie', 'Services', 'Transport']
regions = ['France', 'Allemagne', 'Italie']
n_sectors = len(sectors)
n_regions = len(regions)
n_total = n_sectors * n_regions

# Labels
labels = [f"{r}_{s}" for r in regions for s in sectors]

# Matrice de coefficients techniques (A) — qui achète à qui
np.random.seed(42)
A = np.random.uniform(0.01, 0.08, (n_total, n_total))

# Renforcer les liens intra-régionaux
for r in range(n_regions):
    start = r * n_sectors
    end = start + n_sectors
    A[start:end, start:end] *= 3  # 3× plus d'échanges intra-région

# Renforcer les chaînes logiques
for r in range(n_regions):
    base = r * n_sectors
    A[base + 1, base + 0] *= 2  # Industrie achète à Agriculture
    A[base + 1, base + 2] *= 2  # Industrie achète à Énergie
    A[base + 4, base + 2] *= 2  # Transport achète à Énergie
    A[base + 3, base + 1] *= 1.5  # Services achètent à Industrie

# Normaliser pour stabilité (somme des colonnes < 1)
col_sums = A.sum(axis=0)
A = A / (col_sums * 1.5)

# Production totale par secteur
X = np.random.uniform(10, 100, n_total)  # milliards EUR
X_labels = pd.Series(X, index=labels)

# Demande finale
d = X - A @ X

# Matrice de Leontief
I = np.eye(n_total)
L = np.linalg.inv(I - A)  # Leontief inverse

print("Matrice de coefficients techniques A :")
print(f"  Dimension : {A.shape}")
print(f"  Somme max colonne : {A.sum(axis=0).max():.3f}")
print(f"\nProduction totale : {X.sum():.1f} Mds EUR")
print(f"Demande finale : {d.sum():.1f} Mds EUR")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matrice A
im = axes[0].imshow(A, cmap='YlOrRd', aspect='auto')
axes[0].set_title('Matrice de coefficients techniques (A)')
axes[0].set_xticks(range(0, n_total, n_sectors))
axes[0].set_xticklabels(regions, fontsize=9)
axes[0].set_yticks(range(0, n_total, n_sectors))
axes[0].set_yticklabels(regions, fontsize=9)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Matrice de Leontief
im2 = axes[1].imshow(L, cmap='YlOrRd', aspect='auto')
axes[1].set_title('Inverse de Leontief (L = (I-A)⁻¹)')
axes[1].set_xticks(range(0, n_total, n_sectors))
axes[1].set_xticklabels(regions, fontsize=9)
axes[1].set_yticks(range(0, n_total, n_sectors))
axes[1].set_yticklabels(regions, fontsize=9)
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Lignes de séparation entre régions
for ax in axes:
    for i in range(1, n_regions):
        ax.axhline(y=i * n_sectors - 0.5, color='white', linewidth=1)
        ax.axvline(x=i * n_sectors - 0.5, color='white', linewidth=1)

plt.tight_layout()
plt.show()

## 3. Scénarios de choc climatique

On simule l'impact de différents événements climatiques sur la capacité de production.

In [ ]:
# Scénarios de choc (perte de capacité de production)
scenarios = {
    'Inondation Allemagne (2021)': {
        'description': 'Inondation massive dans la Ruhr — industrie et transport paralysés',
        'shocks': {
            'Allemagne_Industrie': 0.25,
            'Allemagne_Transport': 0.30,
            'Allemagne_Énergie': 0.10,
        }
    },
    'Sécheresse France (2022)': {
        'description': 'Sécheresse record — agriculture et énergie (refroidissement centrales)',
        'shocks': {
            'France_Agriculture': 0.35,
            'France_Énergie': 0.15,
        }
    },
    'Tempête Europe (Xynthia)': {
        'description': 'Tempête hivernale affectant toute l\'Europe de l\'Ouest',
        'shocks': {
            'France_Industrie': 0.10,
            'France_Transport': 0.20,
            'Allemagne_Transport': 0.15,
            'Italie_Transport': 0.10,
        }
    },
    'Canicule Italie (2023)': {
        'description': 'Canicule extrême — agriculture et services (tourisme)',
        'shocks': {
            'Italie_Agriculture': 0.30,
            'Italie_Services': 0.15,
        }
    },
}

results = {}
for scenario_name, scenario in scenarios.items():
    # Vecteur de choc (réduction de demande finale)
    delta_d = np.zeros(n_total)
    for sector_label, loss_frac in scenario['shocks'].items():
        idx = labels.index(sector_label)
        delta_d[idx] = -d[idx] * loss_frac
    
    # Impact via Leontief : ΔX = L × Δd
    delta_X = L @ delta_d
    
    # Pertes directes et indirectes
    direct_loss = -delta_d.sum()
    total_loss = -delta_X.sum()
    indirect_loss = total_loss - direct_loss
    multiplier = total_loss / direct_loss if direct_loss > 0 else 0
    
    results[scenario_name] = {
        'direct': direct_loss,
        'indirect': indirect_loss,
        'total': total_loss,
        'multiplier': multiplier,
        'delta_X': delta_X,
        'description': scenario['description']
    }
    
    print(f"\n--- {scenario_name} ---")
    print(f"  {scenario['description']}")
    print(f"  Perte directe :   {direct_loss:>8.2f} Mds EUR")
    print(f"  Perte indirecte : {indirect_loss:>8.2f} Mds EUR")
    print(f"  Perte totale :    {total_loss:>8.2f} Mds EUR")
    print(f"  Multiplicateur :  {multiplier:.2f}×")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Comparaison directe vs indirecte
ax = axes[0, 0]
scenario_names = list(results.keys())
short_names = ['Inondation\nDE', 'Sécheresse\nFR', 'Tempête\nEurope', 'Canicule\nIT']
direct = [results[s]['direct'] for s in scenario_names]
indirect = [results[s]['indirect'] for s in scenario_names]

x = np.arange(len(scenario_names))
ax.bar(x - 0.2, direct, 0.35, label='Direct', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, indirect, 0.35, label='Indirect', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylabel('Perte (Mds EUR)')
ax.set_title('Pertes directes vs indirectes')
ax.legend()

# 2. Multiplicateurs
ax = axes[0, 1]
multipliers = [results[s]['multiplier'] for s in scenario_names]
colors = ['green' if m < 1.5 else 'orange' if m < 2 else 'red' for m in multipliers]
ax.bar(short_names, multipliers, color=colors, alpha=0.8)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Multiplicateur (total/direct)')
ax.set_title('Effet multiplicateur par scénario')

# 3. Impact par région (inondation DE)
ax = axes[1, 0]
delta_X_flood = results['Inondation Allemagne (2021)']['delta_X']
region_losses = {}
for r_idx, region in enumerate(regions):
    start = r_idx * n_sectors
    end = start + n_sectors
    region_losses[region] = -delta_X_flood[start:end].sum()

ax.bar(region_losses.keys(), region_losses.values(), 
       color=['#2196F3', '#FF9800', '#4CAF50'], alpha=0.8)
ax.set_ylabel('Perte totale (Mds EUR)')
ax.set_title('Inondation DE 2021 — Impact par région')

# 4. Impact par secteur (inondation DE)
ax = axes[1, 1]
sector_losses = {}
for s_idx, sector in enumerate(sectors):
    total = sum(-delta_X_flood[r * n_sectors + s_idx] for r in range(n_regions))
    sector_losses[sector] = total

ax.barh(list(sector_losses.keys()), list(sector_losses.values()), color='steelblue', alpha=0.8)
ax.set_xlabel('Perte totale (Mds EUR)')
ax.set_title('Inondation DE 2021 — Impact par secteur')

plt.tight_layout()
plt.show()

## 4. Dynamique de récupération

Simulation simplifiée de la dynamique post-choc : comment l'économie se rétablit après un événement.

In [ ]:
# Simulation dynamique simplifiée
n_steps = 365  # jours
recovery_rate = 0.005  # 0.5% de récupération par jour

# Choc initial : inondation Allemagne
initial_shock = np.zeros(n_total)
for sector_label, loss_frac in scenarios['Inondation Allemagne (2021)']['shocks'].items():
    idx = labels.index(sector_label)
    initial_shock[idx] = loss_frac

# Simulation jour par jour
capacity = np.ones((n_steps, n_total))
production_loss = np.zeros((n_steps, n_total))

for t in range(n_steps):
    if t == 0:
        capacity[t] = 1 - initial_shock
    else:
        # Récupération exponentielle
        gap = 1 - capacity[t-1]
        capacity[t] = capacity[t-1] + gap * recovery_rate
    
    # Perte de production = (1 - capacité) × production nominale
    production_loss[t] = (1 - capacity[t]) * X

# Agrégation par région
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

days = np.arange(n_steps)
for r_idx, region in enumerate(regions):
    start = r_idx * n_sectors
    end = start + n_sectors
    region_loss = production_loss[:, start:end].sum(axis=1)
    axes[0].plot(days, region_loss, linewidth=2, label=region)

axes[0].set_xlabel('Jours après l\'événement')
axes[0].set_ylabel('Perte de production (Mds EUR/jour)')
axes[0].set_title('Dynamique de récupération par région')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perte cumulée
for r_idx, region in enumerate(regions):
    start = r_idx * n_sectors
    end = start + n_sectors
    cumul_loss = np.cumsum(production_loss[:, start:end].sum(axis=1))
    axes[1].plot(days, cumul_loss, linewidth=2, label=region)

axes[1].set_xlabel('Jours après l\'événement')
axes[1].set_ylabel('Perte cumulée (Mds EUR)')
axes[1].set_title('Perte cumulée de production')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

total_cumul = production_loss.sum()
print(f"Perte totale cumulée sur {n_steps} jours : {total_cumul:.2f} Mds EUR")
print(f"Temps de récupération à 95% : {np.argmax(capacity.min(axis=1) > 0.95)} jours")

plt.tight_layout()
plt.show()

## 5. Matrice de vulnérabilité croisée

Quels secteurs/régions sont les plus vulnérables aux chocs dans d'autres secteurs/régions ?

In [ ]:
# Pour chaque secteur-région, simuler un choc unitaire et mesurer l'impact total
vulnerability_matrix = np.zeros((n_total, n_total))

for i in range(n_total):
    # Choc unitaire sur le secteur i
    delta_d_unit = np.zeros(n_total)
    delta_d_unit[i] = -1.0
    
    # Propagation
    delta_X = L @ delta_d_unit
    vulnerability_matrix[:, i] = -delta_X

# Agrégation par région
region_vuln = np.zeros((n_regions, n_regions))
for r_src in range(n_regions):
    for r_dst in range(n_regions):
        src_start = r_src * n_sectors
        dst_start = r_dst * n_sectors
        region_vuln[r_dst, r_src] = vulnerability_matrix[
            dst_start:dst_start+n_sectors, src_start:src_start+n_sectors
        ].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice région × région
im = axes[0].imshow(region_vuln, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(n_regions))
axes[0].set_xticklabels(regions)
axes[0].set_yticks(range(n_regions))
axes[0].set_yticklabels(regions)
axes[0].set_xlabel('Région source du choc')
axes[0].set_ylabel('Région impactée')
axes[0].set_title('Vulnérabilité croisée inter-régionale')
for i in range(n_regions):
    for j in range(n_regions):
        axes[0].text(j, i, f'{region_vuln[i, j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Secteurs les plus "systémiques" (qui causent le plus de dommages quand ils sont touchés)
systemic_score = vulnerability_matrix.sum(axis=0)
sector_systemic = pd.Series(systemic_score, index=labels).sort_values(ascending=True)

axes[1].barh(range(len(sector_systemic)), sector_systemic.values, color='steelblue', alpha=0.8)
axes[1].set_yticks(range(len(sector_systemic)))
axes[1].set_yticklabels(sector_systemic.index, fontsize=7)
axes[1].set_xlabel('Score systémique (impact total d\'un choc unitaire)')
axes[1].set_title('Secteurs les plus systémiques')

plt.tight_layout()
plt.show()

## 6. Intégration avec CLIMADA (BoARIO)

### API BoARIO (quand disponible)

```python
# 1. Calculer les impacts directs avec CLIMADA
impact = Impact()
impact.calc(exposure, impf_set, hazard)

# 2. Créer les chocs directs
shocks = DirectShocksSet(
    mriot_name='exiobase3',
    exposure_assets=exposure,
    impacted_assets=impact
)

# 3. Lancer la simulation BoARIO
model = BoARIOModel(
    mriot_name='exiobase3',
    direct_shocks=shocks,
    n_timesteps=365,
    recovery_rate=0.005
)
model.run()

# 4. Analyser les résultats
indirect_costs = model.get_indirect_costs()
total_costs = model.get_total_costs()
```

## 7. Synthèse

| Aspect | Détail |
|--------|--------|
| Modèle | BoARIO (Adaptative Regional IO) |
| Données | Tables MRIO (EXIOBASE, WIOD, GTAP) |
| Input | Impacts directs CLIMADA |
| Output | Coûts indirects, multiplicateurs, dynamique de récupération |
| Granularité | Pays × secteur économique |

### Points clés
- Les coûts indirects peuvent dépasser les coûts directs (multiplicateur 1.5-3×)
- L'industrie et les transports sont les secteurs les plus "systémiques"
- Les effets de cascade amplifient les chocs locaux en impacts globaux
- Le temps de récupération dépend de la résilience des chaînes (stocks, substitution)

### Multiplicateurs typiques par péril
| Péril | Multiplicateur | Raison |
|-------|---------------|--------|
| Inondation | 2-3× | Destruction d'infrastructure, longue reconstruction |
| Tempête | 1.5-2× | Perturbation transport, récupération rapide |
| Sécheresse | 1.3-1.8× | Impact diffus, adaptation possible |
| Séisme | 3-5× | Destruction massive, reconstruction longue |

### Prochaines étapes
- Installer BoARIO et charger une table MRIO réelle (EXIOBASE3)
- Coupler avec les impacts CLIMADA des notebooks précédents
- Analyser la vulnérabilité des chaînes d'approvisionnement françaises
- Explorer les stratégies de résilience (diversification, stocks stratégiques)